# Music Annotation Project V9.0A
## Duplicate-Aware Mask-Authority Transfer Audit

This revision fixes the Notebook 06/07 audit problem by grouping all records with the same `source_md5`, inventorying duplicates, and resolving only an unambiguous Notebook 06 → Notebook 07 lineage. It never silently keeps the first or last duplicate.

Canonical identity:
- Notebook 06 files keep the original source stem.
- Notebook 07 supplies the downstream `PageXXX` identity.
- Cross-stage linkage is `source_md5`.

The experiment remains development/shadow evidence only. It does not authorise automatic mask authority.

In [26]:
# PROVENANCE: standard project Colab bootstrap.
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration

In [27]:
# PROVENANCE: paths from Notebooks 06, 07 and 09; V9.0A run controls.
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, re, sys, platform
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
CLEAN_DIR=ROOT/'source'/'clean'
OUTPUT_ROOT=ROOT/'mask_authority_transfer_v9_0a'
PAGE_ALLOWLIST=None
# Established prior mask-authority gate. V8.2b records 0.90 as the exploratory automatic candidate.
ESTABLISHED_YOLO_GATE=0.90
ESTABLISHED_YOLO_GATE_POLICY_ID='V8.2b_AUTO_GE_090_RUN2'
MASK_DILATION=2; CROP_MARGIN=48; GEOMETRY_THRESHOLD=80; BAND_WIDTH=4
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v9_0a_duplicate_aware'
OUT=OUTPUT_ROOT/'runs'/RUN_ID
INPUTS=OUT/'01_inputs'; MASKS=OUT/'02_mask_arms'; OUTPUTS=OUT/'03_corrected_pages'; LOCKS=OUT/'04_locks'; SCORE=OUT/'05_scoring'; VISUAL=OUT/'06_visual_audit'; AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MASKS,OUTPUTS,LOCKS,SCORE,VISUAL,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
ARMS=['reviewed_comparator','raw_yolo','confidence_gated_yolo','reviewer_corrected_yolo']
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
print('Output:',OUT)

Output: /content/drive/MyDrive/MusicAnnotationProject/mask_authority_transfer_v9_0a/runs/20261110_095807_v9_0a_duplicate_aware


## 2. Integrity helpers

In [28]:
# PROVENANCE: shared project integrity helpers.
def require(x,msg):
    if not bool(x):
        try:(AUDIT/'fatal_failure.json').write_text(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'message':str(msg)},indent=2))
        except Exception:pass
        raise RuntimeError(msg)
def md5_file(p):
    h=hashlib.md5()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def sha256_file(p):
    p=Path(p);require(p.exists(),f'Missing file: {p}');h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(p,v):
    p=Path(p);t=Path(str(p)+'.tmp');t.write_text(json.dumps(v,indent=2,sort_keys=True,default=str));t.replace(p)
def read_gray(p):
    a=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE);require(a is not None,f'Unreadable image: {p}');return a
def binary(a):return np.where(a>0,255,0).astype(np.uint8)

## 3. Load, group and diagnose all Notebook 06/07 audits

In [ ]:
# PROVENANCE:
# Adapted from Notebook 09 HITL audit resolution.
#
# PURPOSE:
# Resolve the canonical Notebook 07 PageXXX reviewed roster backwards to the
# corresponding Notebook 06 draft artefacts.
#
# IMPORTANT:
# - Notebook 07 review audits define the eligible reviewed-page population.
# - Notebook 07 `page` defines the canonical V9.0A page identity.
# - Notebook 06 and Notebook 07 filenames are not expected to share stems.
# - Whole-directory source_md5 intersection is not required.
# - Resolution uses, in order:
#       1. exact original-source path
#       2. source content MD5 calculated from the recorded source path
#       3. stored source_md5, where compatible
#       4. unique original-source filename
# - A filename-only match is accepted only when it is unique.
# - Ambiguous matches fail closed.
# - Every attempted match is written to the audit output.


def load_audit_records(folder, audit_type):
    records = []
    failures = []

    for audit_path in sorted(Path(folder).glob("*.json")):
        try:
            record = json.loads(audit_path.read_text())

            if not isinstance(record, dict):
                raise ValueError("Audit root must be a JSON object.")

            record["_audit_path"] = str(audit_path)
            record["_audit_name"] = audit_path.name
            record["_audit_stem"] = audit_path.stem
            record["_audit_type"] = audit_type
            record["_audit_mtime_ns"] = audit_path.stat().st_mtime_ns

            records.append(record)

        except Exception as error:
            failures.append({
                "audit_type": audit_type,
                "audit_path": str(audit_path),
                "error": str(error),
            })

    return records, failures


def normalised_path_string(value):
    """
    Produce a stable comparison string without requiring the path to exist.

    pathlib.resolve() is deliberately not used because historical audit
    records may contain paths that are no longer mounted or have moved.
    """
    if value is None:
        return None

    value = str(value).strip()

    if not value:
        return None

    return value.replace("\\", "/").rstrip("/").casefold()


def recorded_path(record, possible_fields):
    """
    Return the first non-empty recorded path and its field name.
    The path is not required to exist at this point.
    """
    for field in possible_fields:
        value = record.get(field)

        if value is not None and str(value).strip():
            return Path(str(value)), field

    return None, None


def required_existing_path(record, possible_fields, audit_type):
    """
    Resolve the first referenced path which currently exists.
    Fail with all attempted field values if none exists.
    """
    attempted = []

    for field in possible_fields:
        value = record.get(field)

        if value is None or not str(value).strip():
            continue

        path = Path(str(value))
        attempted.append({
            "field": field,
            "path": str(path),
            "exists": path.exists(),
        })

        if path.exists():
            return path, field

    raise RuntimeError(
        f"{audit_type} audit has no existing file for fields "
        f"{possible_fields}: {record.get('_audit_path')}. "
        f"Attempted: {attempted}"
    )


def audit_source_path(record, audit_type):
    """
    Extract the original source-page path recorded by each notebook.

    Notebook 06 normally records `source_file`.
    Notebook 07 normally records `source`.
    Aliases are included only to expose historical schema variants.
    """
    if audit_type == "draft":
        fields = [
            "source_file",
            "source",
            "input_file",
            "input_path",
        ]
    elif audit_type == "review":
        fields = [
            "source",
            "source_file",
            "original_source",
            "input_file",
            "input_path",
        ]
    else:
        raise ValueError(audit_type)

    return recorded_path(record, fields)


def safe_recorded_md5(record):
    value = record.get("source_md5")

    if value is None:
        return None

    value = str(value).strip().lower()

    return value or None


def existing_source_md5(record, audit_type):
    """
    Calculate MD5 from the actual recorded source file when that file still
    exists. This is stronger evidence than trusting a stored hash field.
    """
    path, field = audit_source_path(record, audit_type)

    if path is None or not path.exists():
        return None, path, field

    return md5_file(path).lower(), path, field


def stable_draft_signature(record):
    """
    Semantic identity for duplicate Notebook 06 audit records.

    Records with the same signature refer to the same source, model,
    draft mask and confidence map. Duplicate JSON copies of the same
    semantic record can therefore be collapsed safely.
    """
    fields = [
        "source_file",
        "source_md5",
        "draft_mask",
        "confidence_map",
        "model_path",
        "model_md5",
        "prediction_confidence",
        "shape",
    ]

    payload = {
        field: record.get(field)
        for field in fields
    }

    return object_hash(payload)


def stable_review_signature(record):
    """
    Semantic identity for duplicate Notebook 07 review records.
    """
    fields = [
        "page",
        "source",
        "source_md5",
        "aligned_output",
        "corrected_mask_output",
    ]

    payload = {
        field: record.get(field)
        for field in fields
    }

    return object_hash(payload)


def collapse_identical_records(records, audit_type, context):
    """
    Collapse only semantically identical replicas.

    Conflicting records are not resolved by first, last, newest or oldest.
    """
    if len(records) == 1:
        return records[0], "unique", 1

    signature_function = (
        stable_draft_signature
        if audit_type == "draft"
        else stable_review_signature
    )

    groups = {}

    for record in records:
        signature = signature_function(record)
        groups.setdefault(signature, []).append(record)

    if len(groups) != 1:
        raise RuntimeError(
            f"{context}: conflicting {audit_type} audit records. "
            f"Candidates: "
            f"{[record['_audit_path'] for record in records]}"
        )

    replicas = next(iter(groups.values()))

    # Selection among identical semantic replicas is immaterial.
    # Sorting makes the choice deterministic.
    chosen = sorted(
        replicas,
        key=lambda record: record["_audit_path"],
    )[0]

    return chosen, "identical_replicas_collapsed", len(replicas)


# ------------------------------------------------------------------
# Load all audit records.
# ------------------------------------------------------------------

DRAFT_AUDITS, draft_parse_failures = load_audit_records(
    DRAFT_AUDIT_DIR,
    "draft",
)

REVIEW_AUDITS, review_parse_failures = load_audit_records(
    REVIEW_AUDIT_DIR,
    "review",
)

audit_parse_failures = (
    draft_parse_failures
    + review_parse_failures
)

if audit_parse_failures:
    pd.DataFrame(audit_parse_failures).to_csv(
        AUDIT / "audit_parse_failures.csv",
        index=False,
    )

    raise RuntimeError(
        f"{len(audit_parse_failures)} HITL audit record(s) "
        f"could not be read. See "
        f"{AUDIT / 'audit_parse_failures.csv'}"
    )

require(
    len(DRAFT_AUDITS) > 0,
    f"No Notebook 06 draft audits found in {DRAFT_AUDIT_DIR}",
)

require(
    len(REVIEW_AUDITS) > 0,
    f"No Notebook 07 review audits found in {REVIEW_AUDIT_DIR}",
)

print("Notebook 06 draft audits:", len(DRAFT_AUDITS))
print("Notebook 07 review audits:", len(REVIEW_AUDITS))


# ------------------------------------------------------------------
# Build multiple Notebook 06 lookup indexes.
# ------------------------------------------------------------------

drafts_by_recorded_path = {}
drafts_by_actual_md5 = {}
drafts_by_stored_md5 = {}
drafts_by_source_name = {}
draft_inventory_rows = []


for draft in DRAFT_AUDITS:
    source_path, source_field = audit_source_path(
        draft,
        "draft",
    )

    source_path_key = normalised_path_string(source_path)

    actual_md5 = None

    if source_path is not None and source_path.exists():
        actual_md5 = md5_file(source_path).lower()

    stored_md5 = safe_recorded_md5(draft)

    source_name = (
        source_path.name.casefold()
        if source_path is not None
        else None
    )

    if source_path_key:
        drafts_by_recorded_path.setdefault(
            source_path_key,
            [],
        ).append(draft)

    if actual_md5:
        drafts_by_actual_md5.setdefault(
            actual_md5,
            [],
        ).append(draft)

    if stored_md5:
        drafts_by_stored_md5.setdefault(
            stored_md5,
            [],
        ).append(draft)

    if source_name:
        drafts_by_source_name.setdefault(
            source_name,
            [],
        ).append(draft)

    draft_inventory_rows.append({
        "draft_audit_path": draft["_audit_path"],
        "recorded_source_field": source_field,
        "recorded_source_path": (
            str(source_path)
            if source_path is not None
            else None
        ),
        "recorded_source_exists": (
            source_path.exists()
            if source_path is not None
            else False
        ),
        "actual_source_md5": actual_md5,
        "stored_source_md5": stored_md5,
        "source_filename": (
            source_path.name
            if source_path is not None
            else None
        ),
        "draft_mask": draft.get("draft_mask"),
        "confidence_map": draft.get("confidence_map"),
        "model_path": draft.get("model_path"),
        "model_md5": draft.get("model_md5"),
        "prediction_confidence": (
            draft.get("prediction_confidence")
        ),
    })


pd.DataFrame(draft_inventory_rows).to_csv(
    AUDIT / "notebook06_draft_audit_inventory.csv",
    index=False,
)


# ------------------------------------------------------------------
# Resolve Notebook 07 reviews backwards to Notebook 06 records.
# ------------------------------------------------------------------

resolved_rows = []
resolution_attempt_rows = []
unresolved_rows = []


for review in REVIEW_AUDITS:
    page = str(review.get("page", "")).strip()

    if not page:
        page = Path(review["_audit_path"]).stem

    if not re.fullmatch(r"Page\d+", page, flags=re.IGNORECASE):
        unresolved_rows.append({
            "page": page,
            "review_audit_path": review["_audit_path"],
            "reason": "invalid_notebook07_page_identity",
        })
        continue

    review_source_path, review_source_field = audit_source_path(
        review,
        "review",
    )

    review_source_path_key = normalised_path_string(
        review_source_path
    )

    review_stored_md5 = safe_recorded_md5(review)

    review_actual_md5 = None

    if (
        review_source_path is not None
        and review_source_path.exists()
    ):
        review_actual_md5 = md5_file(
            review_source_path
        ).lower()

    review_source_name = (
        review_source_path.name.casefold()
        if review_source_path is not None
        else None
    )

    candidate_sets = []

    # 1. Exact recorded original-source path.
    if (
        review_source_path_key
        and review_source_path_key in drafts_by_recorded_path
    ):
        candidate_sets.append({
            "method": "exact_recorded_source_path",
            "strength": 1,
            "candidates": drafts_by_recorded_path[
                review_source_path_key
            ],
        })

    # 2. Actual hash calculated from the review's recorded source file.
    if (
        review_actual_md5
        and review_actual_md5 in drafts_by_actual_md5
    ):
        candidate_sets.append({
            "method": "actual_source_file_md5",
            "strength": 2,
            "candidates": drafts_by_actual_md5[
                review_actual_md5
            ],
        })

    # 3. Review's actual source hash against stored draft hashes.
    if (
        review_actual_md5
        and review_actual_md5 in drafts_by_stored_md5
    ):
        candidate_sets.append({
            "method": (
                "review_actual_md5_to_draft_stored_md5"
            ),
            "strength": 3,
            "candidates": drafts_by_stored_md5[
                review_actual_md5
            ],
        })

    # 4. Stored review hash against actual Notebook 06 source hashes.
    if (
        review_stored_md5
        and review_stored_md5 in drafts_by_actual_md5
    ):
        candidate_sets.append({
            "method": (
                "review_stored_md5_to_draft_actual_md5"
            ),
            "strength": 4,
            "candidates": drafts_by_actual_md5[
                review_stored_md5
            ],
        })

    # 5. Stored hash on both sides.
    if (
        review_stored_md5
        and review_stored_md5 in drafts_by_stored_md5
    ):
        candidate_sets.append({
            "method": "stored_source_md5",
            "strength": 5,
            "candidates": drafts_by_stored_md5[
                review_stored_md5
            ],
        })

    # 6. Unique original source filename.
    # This is only a fallback because directories may have moved.
    if (
        review_source_name
        and review_source_name in drafts_by_source_name
    ):
        candidate_sets.append({
            "method": "unique_original_source_filename",
            "strength": 6,
            "candidates": drafts_by_source_name[
                review_source_name
            ],
        })

    for candidate_set in candidate_sets:
        for candidate in candidate_set["candidates"]:
            resolution_attempt_rows.append({
                "page": page,
                "review_audit_path": review["_audit_path"],
                "review_source_field": review_source_field,
                "review_source_path": (
                    str(review_source_path)
                    if review_source_path is not None
                    else None
                ),
                "review_source_exists": (
                    review_source_path.exists()
                    if review_source_path is not None
                    else False
                ),
                "review_actual_md5": review_actual_md5,
                "review_stored_md5": review_stored_md5,
                "resolution_method": candidate_set["method"],
                "resolution_strength": candidate_set["strength"],
                "draft_audit_candidate": (
                    candidate["_audit_path"]
                ),
            })

    if not candidate_sets:
        unresolved_rows.append({
            "page": page,
            "review_audit_path": review["_audit_path"],
            "review_source_path": (
                str(review_source_path)
                if review_source_path is not None
                else None
            ),
            "review_actual_md5": review_actual_md5,
            "review_stored_md5": review_stored_md5,
            "reason": "no_notebook06_candidate_found",
        })
        continue

    # Work from strongest to weakest method.
    candidate_sets = sorted(
        candidate_sets,
        key=lambda item: item["strength"],
    )

    chosen_draft = None
    chosen_method = None
    chosen_replica_status = None
    chosen_replica_count = None
    conflicting_method_errors = []

    for candidate_set in candidate_sets:
        # Remove the same audit found more than once.
        unique_candidates = {
            candidate["_audit_path"]: candidate
            for candidate in candidate_set["candidates"]
        }

        candidates = list(unique_candidates.values())

        try:
            (
                chosen_draft,
                chosen_replica_status,
                chosen_replica_count,
            ) = collapse_identical_records(
                candidates,
                "draft",
                (
                    f"{page} via "
                    f"{candidate_set['method']}"
                ),
            )

            chosen_method = candidate_set["method"]
            break

        except RuntimeError as error:
            conflicting_method_errors.append({
                "method": candidate_set["method"],
                "error": str(error),
            })

    if chosen_draft is None:
        unresolved_rows.append({
            "page": page,
            "review_audit_path": review["_audit_path"],
            "review_source_path": (
                str(review_source_path)
                if review_source_path is not None
                else None
            ),
            "review_actual_md5": review_actual_md5,
            "review_stored_md5": review_stored_md5,
            "reason": "all_candidate_methods_ambiguous",
            "details": json.dumps(
                conflicting_method_errors,
                sort_keys=True,
            ),
        })
        continue

    try:
        observed_path, observed_field = required_existing_path(
            review,
            [
                "aligned_output",
                "aligned_page",
                "observed_output",
            ],
            "Notebook 07 review",
        )

        reviewed_mask_path, reviewed_mask_field = (
            required_existing_path(
                review,
                [
                    "corrected_mask_output",
                    "corrected_mask",
                    "mask_output",
                ],
                "Notebook 07 review",
            )
        )

        raw_mask_path, raw_mask_field = (
            required_existing_path(
                chosen_draft,
                [
                    "draft_mask",
                    "draft_mask_path",
                ],
                "Notebook 06 draft",
            )
        )

        confidence_path, confidence_field = (
            required_existing_path(
                chosen_draft,
                [
                    "confidence_map",
                    "confidence_map_path",
                ],
                "Notebook 06 draft",
            )
        )

        draft_source_path, draft_source_field = (
            required_existing_path(
                chosen_draft,
                [
                    "source_file",
                    "source",
                    "input_file",
                    "input_path",
                ],
                "Notebook 06 draft",
            )
        )

        observed = read_gray(observed_path)
        raw_mask = read_gray(raw_mask_path)
        reviewed_mask = read_gray(reviewed_mask_path)

        confidence = np.load(
            confidence_path
        ).astype(np.float32)

        require(
            observed.shape
            == raw_mask.shape
            == reviewed_mask.shape
            == confidence.shape,
            (
                f"{page}: shape mismatch: "
                f"observed={observed.shape}, "
                f"raw={raw_mask.shape}, "
                f"reviewed={reviewed_mask.shape}, "
                f"confidence={confidence.shape}"
            ),
        )

        require(
            np.isfinite(confidence).all(),
            f"{page}: confidence map contains non-finite values.",
        )

        draft_actual_md5 = md5_file(
            draft_source_path
        ).lower()

        draft_stored_md5 = safe_recorded_md5(
            chosen_draft
        )

        # The Notebook 06 source file must agree with its own stored hash
        # whenever that stored field exists.
        if draft_stored_md5 is not None:
            require(
                draft_actual_md5 == draft_stored_md5,
                (
                    f"{page}: selected Notebook 06 audit has "
                    f"a stale source_md5. "
                    f"Stored={draft_stored_md5}; "
                    f"actual={draft_actual_md5}; "
                    f"audit={chosen_draft['_audit_path']}"
                ),
            )

        # If Notebook 07's original source still exists, require its bytes
        # to agree with the selected Notebook 06 original source.
        if review_actual_md5 is not None:
            require(
                draft_actual_md5 == review_actual_md5,
                (
                    f"{page}: the selected Notebook 06 and "
                    f"Notebook 07 original source files differ. "
                    f"Notebook06={draft_actual_md5}; "
                    f"Notebook07={review_actual_md5}"
                ),
            )

        resolved_rows.append({
            "page": page,

            # Canonical source identity for this resolved lineage.
            "source_md5": draft_actual_md5,

            "original_source_path": str(
                draft_source_path
            ),
            "observed_path": str(observed_path),
            "raw_yolo_mask_path": str(raw_mask_path),
            "confidence_map_path": str(confidence_path),
            "reviewed_mask_path": str(
                reviewed_mask_path
            ),

            "draft_audit_path": (
                chosen_draft["_audit_path"]
            ),
            "review_audit_path": (
                review["_audit_path"]
            ),

            "resolution_method": chosen_method,
            "draft_replica_status": (
                chosen_replica_status
            ),
            "draft_replica_count": (
                chosen_replica_count
            ),

            "review_recorded_source_path": (
                str(review_source_path)
                if review_source_path is not None
                else None
            ),
            "review_actual_source_md5": (
                review_actual_md5
            ),
            "review_stored_source_md5": (
                review_stored_md5
            ),

            "draft_actual_source_md5": (
                draft_actual_md5
            ),
            "draft_stored_source_md5": (
                draft_stored_md5
            ),

            "observed_field": observed_field,
            "reviewed_mask_field": (
                reviewed_mask_field
            ),
            "raw_mask_field": raw_mask_field,
            "confidence_field": confidence_field,
            "draft_source_field": (
                draft_source_field
            ),

            "draft_model_path": (
                chosen_draft.get("model_path")
            ),
            "draft_model_md5": (
                chosen_draft.get("model_md5")
            ),
            "draft_prediction_confidence": (
                chosen_draft.get(
                    "prediction_confidence"
                )
            ),
        })

    except Exception as error:
        unresolved_rows.append({
            "page": page,
            "review_audit_path": review["_audit_path"],
            "draft_audit_path": (
                chosen_draft["_audit_path"]
                if chosen_draft is not None
                else None
            ),
            "resolution_method": chosen_method,
            "reason": "resolved_candidate_failed_validation",
            "details": str(error),
        })


# ------------------------------------------------------------------
# Write diagnostics before deciding whether the roster is usable.
# ------------------------------------------------------------------

attempt_columns = [
    "page",
    "review_audit_path",
    "review_source_field",
    "review_source_path",
    "review_source_exists",
    "review_actual_md5",
    "review_stored_md5",
    "resolution_method",
    "resolution_strength",
    "draft_audit_candidate",
]

pd.DataFrame(
    resolution_attempt_rows,
    columns=attempt_columns,
).to_csv(
    AUDIT / "notebook07_to_notebook06_resolution_attempts.csv",
    index=False,
)

unresolved_columns = [
    "page",
    "review_audit_path",
    "draft_audit_path",
    "review_source_path",
    "review_actual_md5",
    "review_stored_md5",
    "resolution_method",
    "reason",
    "details",
]

unresolved = pd.DataFrame(
    unresolved_rows,
    columns=unresolved_columns,
)

unresolved.to_csv(
    AUDIT / "unresolved_review_lineages.csv",
    index=False,
)


# ------------------------------------------------------------------
# Construct the canonical V9.0A roster.
# ------------------------------------------------------------------

resolution = pd.DataFrame(resolved_rows)

require(
    not resolution.empty,
    (
        "No Notebook 07 reviewed pages could be resolved "
        "back to valid Notebook 06 draft artefacts. "
        f"Inspect {AUDIT / 'notebook07_to_notebook06_resolution_attempts.csv'} "
        f"and {AUDIT / 'unresolved_review_lineages.csv'}."
    ),
)

require(
    not resolution["page"].duplicated().any(),
    (
        "Duplicate Notebook 07 PageXXX identities remain. "
        "Inspect the resolution audit."
    ),
)

require(
    not resolution["source_md5"].duplicated().any(),
    (
        "More than one canonical PageXXX resolves to the same "
        "original Notebook 06 source. This requires explicit "
        "review rather than automatic selection."
    ),
)


# Optional PageXXX roster restriction.
if PAGE_ALLOWLIST is not None:
    requested = {
        str(page)
        for page in PAGE_ALLOWLIST
    }

    available = set(
        resolution["page"].astype(str)
    )

    missing = sorted(requested - available)

    require(
        not missing,
        (
            "Allowlisted PageXXX values do not have complete "
            f"Notebook 06/07 lineage: {missing}"
        ),
    )

    resolution = resolution[
        resolution["page"].astype(str).isin(
            requested
        )
    ].copy()


resolution = (
    resolution
    .sort_values("page")
    .reset_index(drop=True)
)

resolution.to_csv(
    AUDIT / "hitl_input_resolution.csv",
    index=False,
)


# Keep unresolved review records visible, but do not silently include them.
summary = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook06_draft_audits": len(
        DRAFT_AUDITS
    ),
    "notebook07_review_audits": len(
        REVIEW_AUDITS
    ),
    "resolved_pages": len(resolution),
    "unresolved_review_records": len(
        unresolved
    ),
    "resolution_methods": (
        resolution["resolution_method"]
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "canonical_page_identity": (
        "Notebook 07 page / PageXXX"
    ),
    "cross_stage_filename_stem_join": False,
    "whole_directory_md5_intersection_required": False,
}

atomic_json(
    AUDIT / "hitl_resolution_summary.json",
    summary,
)

print(json.dumps(summary, indent=2))

display(
    resolution[
        [
            "page",
            "resolution_method",
            "original_source_path",
            "observed_path",
            "raw_yolo_mask_path",
            "confidence_map_path",
            "reviewed_mask_path",
            "draft_replica_status",
        ]
    ]
)

Notebook 06 draft audits: 173
Notebook 07 review audits: 173


## 4. Resolve one authoritative record per stage without first/last guessing

In [ ]:
# PROVENANCE: NEW V9.0A duplicate resolution. Identical replicas may collapse; conflicting valid records stop.
def resolve_group(records,kind,h):
    valid=[];invalid=[]
    for r in records:
        ok,missing=valid_record(r,kind)
        (valid if ok else invalid).append((r,missing))
    require(valid,f'{h}: no valid {kind} audit; invalid={[(r["_audit_path"],m) for r,m in invalid]}')
    by_sig={}
    for r,_ in valid:by_sig.setdefault(record_signature(r,kind),[]).append(r)
    if len(by_sig)==1:
        replicas=next(iter(by_sig.values()))
        # Deterministic choice only among semantically identical records.
        chosen=sorted(replicas,key=lambda r:r['_audit_path'])[0]
        status='unique' if len(replicas)==1 else 'identical_replicas_collapsed'
        return chosen,status,len(replicas)
    raise RuntimeError(f'{h}: conflicting valid {kind} audits. Inspect {AUDIT/"duplicate_source_md5_audits.csv"}; no record was selected.')

resolved=[];decisions=[]
for h in paired:
    d,ds,dn=resolve_group(draft_groups[h],'draft',h);r,rs,rn=resolve_group(review_groups[h],'review',h)
    page=str(r.get('page','')).strip() or Path(r['_audit_path']).stem
    require(re.fullmatch(r'Page\d+',page,re.I),f'{h}: invalid Notebook 07 page identity {page!r}')
    sp=Path(d['source_file']);rp=Path(d['draft_mask']);cp=Path(d['confidence_map']);op=Path(r['aligned_output']);mp=Path(r['corrected_mask_output'])
    require(md5_file(sp).lower()==h,f'{page}: Notebook 06 source hash mismatch')
    if r.get('source') and Path(r['source']).exists():require(md5_file(Path(r['source'])).lower()==h,f'{page}: Notebook 07 source hash mismatch')
    obs=read_gray(op);raw=read_gray(rp);reviewed=read_gray(mp);conf=np.load(cp).astype(np.float32)
    require(obs.shape==raw.shape==reviewed.shape==conf.shape,f'{page}: shape mismatch')
    require(np.isfinite(conf).all(),f'{page}: non-finite confidence')
    resolved.append({'page':page,'source_md5':h,'original_source_path':str(sp),'observed_path':str(op),'raw_yolo_mask_path':str(rp),'confidence_map_path':str(cp),'reviewed_mask_path':str(mp),'draft_audit_path':d['_audit_path'],'review_audit_path':r['_audit_path'],'draft_resolution':ds,'review_resolution':rs,'draft_replica_count':dn,'review_replica_count':rn,'draft_prediction_confidence':d.get('prediction_confidence'),'draft_model_path':d.get('model_path'),'draft_model_md5':d.get('model_md5')})
    decisions.append({'source_md5':h,'page':page,'draft_status':ds,'draft_chosen':d['_audit_path'],'review_status':rs,'review_chosen':r['_audit_path']})
resolution=pd.DataFrame(resolved_rows).sort_values('page').reset_index(drop=True)
require(not resolution.page.duplicated().any(),'Duplicate PageXXX identities remain after source-hash resolution.')
if PAGE_ALLOWLIST is not None:
    missing=sorted(set(PAGE_ALLOWLIST)-set(resolution.page));require(not missing,f'Allowlisted pages missing: {missing}');resolution=resolution[resolution.page.isin(PAGE_ALLOWLIST)].copy()
require(len(resolution)>0,'No pages remain.')
resolution.to_csv(AUDIT/'hitl_input_resolution.csv',index=False);pd.DataFrame(decisions).to_csv(AUDIT/'audit_resolution_decisions.csv',index=False)
display(resolution[['page','source_md5','resolution_method','draft_replica_status','draft_replica_count','raw_yolo_mask_path','confidence_map_path','reviewed_mask_path','draft_audit_path','review_audit_path']])

## 5. Validate established gate and canonical correction functions

In [ ]:
# PROVENANCE: V8.2b establishes AUTO_THRESHOLD=0.900 and MIN_PRINT_RUN=2 as its exploratory automatic candidate; V9.0A inherits the threshold as a declared experiment input.
require(0.5<=float(ESTABLISHED_YOLO_GATE)<=1.0,'Invalid established gate')
require(str(ESTABLISHED_YOLO_GATE_POLICY_ID).strip(),'Missing gate policy ID')
rawvals=pd.to_numeric(resolution.draft_prediction_confidence,errors='coerce').dropna().unique().tolist()
require(len(rawvals)<=1,f'Notebook 06 raw thresholds disagree: {rawvals}')
raw_threshold=float(rawvals[0]) if rawvals else None
require(raw_threshold is None or ESTABLISHED_YOLO_GATE>raw_threshold,'Gated threshold must exceed raw draft threshold.')
atomic_json(AUDIT/'yolo_gate_identity.json',{'gate':ESTABLISHED_YOLO_GATE,'policy_id':ESTABLISHED_YOLO_GATE_POLICY_ID,'not_optimised_in_v9_0a':True,'raw_draft_threshold':raw_threshold})

def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0];require(np.array_equal(out[mask==0],original[mask==0]),'outside-mask drift');return out
def prepare_mask(seed,dilation=MASK_DILATION):
    m=binary(seed)
    if dilation>0:m=cv2.dilate(m,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return m
def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False;merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop();rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
                else:rest.append((bx0,by0,bx1,by1))
            boxes=rest;merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes
def telea_reconstruct(obs,mask):
    result=obs.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=obs[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1];cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA);local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0]
    return immutable(obs,result,mask)
def background_surface(obs,mask,sigma=45.0):
    v=(mask==0)&(obs>=200);num=cv2.GaussianBlur(obs.astype(np.float32)*v,(0,0),sigma);den=cv2.GaussianBlur(v.astype(np.float32),(0,0),sigma);fallback=cv2.GaussianBlur(obs,(0,0),sigma);return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
def support_maps(obs,mask):
    dark=(obs<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0;n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);cc=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40:cc[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8);return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(cc.astype(np.uint8),k)>0
def geometry_t80(obs,mask,telea):
    paper=background_surface(obs,mask);norm=cv2.divide(obs,np.maximum(paper,1),scale=255);h,v,c=support_maps(obs,mask);m=mask>0;pres=m&(h|v|c)&(obs<200);rem=m&~pres&(obs>=GEOMETRY_THRESHOLD)&(norm>=GEOMETRY_THRESHOLD);amb=m&~pres&~rem;result=obs.copy();result[rem]=paper[rem];result[amb]=telea[amb];result[pres]=obs[pres];return immutable(obs,result,mask),{'preserve':pres,'remove':rem,'ambiguous':amb}

## 6. Execute arms and lock outputs

In [ ]:
# PROVENANCE: V9.0A paired orchestration using canonical dilation-2, Telea and geometry_t80 functions.
runtime={};mr=[];orows=[];ed=[]
for _,r in resolution.iterrows():
    p=r.page;obs=read_gray(r.observed_path);raw=binary(read_gray(r.raw_yolo_mask_path));rev=binary(read_gray(r.reviewed_mask_path));conf=np.load(r.confidence_map_path).astype(np.float32);gated=np.where((raw>0)&(conf>=ESTABLISHED_YOLO_GATE),255,0).astype(np.uint8)
    arms={'reviewed_comparator':rev,'raw_yolo':raw,'confidence_gated_yolo':gated,'reviewer_corrected_yolo':rev.copy()};runtime[p]={}
    ed.append({'page':p,'reviewer_additions':int(((rev>0)&(raw==0)).sum()),'reviewer_erasures':int(((rev==0)&(raw>0)).sum()),'xor':int(((rev>0)^(raw>0)).sum())})
    for arm,seed in arms.items():
        mask=prepare_mask(seed);telea=telea_reconstruct(obs,mask);corr,routes=geometry_t80(obs,mask,telea);d=OUTPUTS/arm/p;d.mkdir(parents=True,exist_ok=False);md=MASKS/arm;md.mkdir(parents=True,exist_ok=True)
        cv2.imwrite(str(md/f'{p}.png'),seed)
        for n,a in [('observed',obs),('seed',seed),('prepared_mask',mask),('corrected',corr),('preserve',routes['preserve'].astype(np.uint8)*255),('remove',routes['remove'].astype(np.uint8)*255),('ambiguous',routes['ambiguous'].astype(np.uint8)*255)]:cv2.imwrite(str(d/f'{n}.png'),a)
        require(np.array_equal(corr[mask==0],obs[mask==0]),f'{p}/{arm}: drift')
        runtime[p][arm]={'seed':seed,'mask':mask,'corrected':corr}
        mr.append({'page':p,'arm':arm,'seed_pixels':int((seed>0).sum()),'prepared_pixels':int((mask>0).sum()),'seed_hash':array_hash(seed),'mask_hash':array_hash(mask)})
        orows.append({'page':p,'arm':arm,'corrected_hash':array_hash(corr),'outside_drift':0,'changed_pixels':int((corr!=obs).sum())})
pd.DataFrame(mr).to_csv(LOCKS/'mask_arm_manifest.csv',index=False);pd.DataFrame(orows).to_csv(LOCKS/'corrected_output_manifest.csv',index=False);pd.DataFrame(ed).to_csv(SCORE/'reviewer_edits.csv',index=False)
lock={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'pages':sorted(runtime),
    'hitl_resolution_sha256':sha256_file(AUDIT/'hitl_input_resolution.csv'),
   # 'duplicate_inventory_sha256':sha256_file(AUDIT/'duplicate_source_md5_audits.csv'),
    'mask_manifest_sha256':sha256_file(LOCKS/'mask_arm_manifest.csv'),
    'output_manifest_sha256':sha256_file(LOCKS/'corrected_output_manifest.csv'),
    'truth_loaded':False
}
lock['lock_id']=object_hash(lock);
atomic_json(LOCKS/'prediction_and_output_lock.json',lock);
print('Lock:',lock['lock_id'])

## 7. Post-lock comparison, optional truth, visuals and verdict

In [ ]:
# PROVENANCE: NEW V9.0A analysis after output lock.
def mm(c,r):
    c=c>0;r=r>0;i=int((c&r).sum());u=int((c|r).sum());return {'iou':i/max(u,1),'fp':int((c&~r).sum()),'fn':int((~c&r).sum()),'xor':int((c^r).sum())}
rows=[]
for p,a in runtime.items():
    rm=a['reviewed_comparator']['seed'];ro=a['reviewed_comparator']['corrected']
    for arm,z in a.items():
        q=mm(z['seed'],rm);q.update({'page':p,'arm':arm,'corrected_xor':int((z['corrected']!=ro).sum()),'corrected_mae':float(np.abs(z['corrected'].astype(np.int16)-ro.astype(np.int16)).mean())});rows.append(q)
paired_df=pd.DataFrame(rows);paired_df.to_csv(SCORE/'paired_evidence.csv',index=False);paired_df.groupby('arm',as_index=False).agg(pages=('page','nunique'),mean_iou=('iou','mean'),fp=('fp','sum'),fn=('fn','sum'),mask_xor=('xor','sum'),corrected_xor=('corrected_xor','sum')).to_csv(SCORE/'arm_summary.csv',index=False)
clean={p.stem:p for p in CLEAN_DIR.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if CLEAN_DIR.exists() else {};tr=[]
for p,a in runtime.items():
    if p not in clean:continue
    truth=read_gray(clean[p])
    for arm,z in a.items():
        diff=np.abs(truth.astype(np.float32)-z['corrected'].astype(np.float32));tr.append({'page':p,'arm':arm,'whole_mae':float(diff.mean()),'inside_mask_mae':float(diff[z['mask']>0].mean()) if (z['mask']>0).any() else np.nan})
if tr:pd.DataFrame(tr).to_csv(SCORE/'postlock_clean_scores.csv',index=False)
q=paired_df[paired_df.arm.isin(['raw_yolo','confidence_gated_yolo'])].copy();q['priority']=q.corrected_xor+5*q.fp+3*q.fn;q.sort_values('priority',ascending=False).to_csv(SCORE/'visual_queue.csv',index=False)
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0A-duplicate-aware','verdict':'PASS_EVIDENCE_READY_FOR_REVIEW','lock_id':lock['lock_id'],'automatic_mask_authority_authorised':False,'duplicate_resolution':'only unique or semantically identical replica groups accepted; conflicting valid groups stop'};atomic_json(OUT/'v9_0a_verdict.json',verdict);print(json.dumps(verdict,indent=2))

## 8. Final Upload Package

In [ ]:
# PROVENANCE: standard Final Upload Package.
required=[OUT/'v9_0a_verdict.json',AUDIT/'all_paired_audit_candidates.csv',AUDIT/'duplicate_source_md5_audits.csv',AUDIT/'audit_resolution_decisions.csv',AUDIT/'hitl_input_resolution.csv',AUDIT/'yolo_gate_identity.json',LOCKS/'mask_arm_manifest.csv',LOCKS/'corrected_output_manifest.csv',LOCKS/'prediction_and_output_lock.json',SCORE/'reviewer_edits.csv',SCORE/'paired_evidence.csv',SCORE/'arm_summary.csv',SCORE/'visual_queue.csv']
if (SCORE/'postlock_clean_scores.csv').exists():required.append(SCORE/'postlock_clean_scores.csv')
for p in required:require(p.exists(),f'Missing upload artefact: {p}')
(UPLOAD/'README.txt').write_text('V9.0A duplicate-aware mask-authority audit. Notebook 06/07 joined by source_md5. Duplicate semantic replicas collapse; conflicting valid records stop. Research evidence only.\n')
rows=[]
for src in required:
    dst=UPLOAD/src.name;shutil.copy2(src,dst);rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['file'] for x in rows))
zp=UPLOAD/'58_V9_0A_upload_bundle.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p!=zp:z.write(p,p.name)
require(zp.exists() and zp.stat().st_size>0,'Bundle failed');print('Bundle:',zp)